# Profiling

Build a measurement workflow that moves from symptoms to evidence about bottlenecks.

## Objectives

Define a reproducible workload, collect appropriate evidence, and avoid perturbing or over-interpreting measurements.

## Background

Profilers add overhead and expose different layers of a system; useful conclusions require a stable baseline and a focused question.

## Prediction

The first workload will contain three explicit phases:

1. transform a sequence of integers using repeated arithmetic and bitwise operations;
2. build a histogram from the transformed values;
3. calculate a weighted checksum.

All three phases traverse the same number of elements, but the transformation performs several rounds of arithmetic per element. It should therefore account for the largest fraction of unprofiled runtime.

The histogram and checksum phases should remain visible but secondary. Their relative cost cannot be inferred from operation counts alone because Python integer arithmetic, list access, loop mechanics, and memory allocation all contribute to elapsed time.

Before collecting a profile, the workload must have:

- deterministic input and output;
- a bounded runtime;
- fixed CPU affinity;
- warm-up runs;
- repeated baseline measurements.

A deterministic function-level profiler such as `cProfile` should later attribute most cumulative time to the transformation phase. However, the profiled runtime should be longer than the unprofiled baseline because profiling records Python call events. The profile can therefore identify where observed runtime is attributed, but its timing should not be treated as an unperturbed measurement of production performance.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
import os
from contextlib import contextmanager
from pprint import pprint

import pandas as pd

from common.benchmark import benchmark_callable


PROFILE_CPU = 15

print(f"Available CPUs: {sorted(os.sched_getaffinity(0))}")
print(f"Selected profiling CPU: {PROFILE_CPU}")

if PROFILE_CPU not in os.sched_getaffinity(0):
    raise RuntimeError(f"CPU {PROFILE_CPU} is not available to the current process")


@contextmanager
def pinned_to_cpu(cpu_id: int):
    original_affinity = os.sched_getaffinity(0)

    try:
        os.sched_setaffinity(0, {cpu_id})
        yield
    finally:
        os.sched_setaffinity(0, original_affinity)


with pinned_to_cpu(PROFILE_CPU):
    print(f"Temporary affinity: {sorted(os.sched_getaffinity(0))}")

Available CPUs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Selected profiling CPU: 15
Temporary affinity: [15]


### Controlled profiling workload

The workload below is intentionally divided into named Python functions. This will later allow a function-level profiler to attribute time to meaningful phases rather than to one monolithic function.

The input is constructed once and reused. Each measured invocation:

1. allocates and fills a transformed list;
2. builds a 256-bin histogram;
3. calculates a weighted checksum.

The returned values prevent the final results from being semantically unused and provide simple correctness checks.

This is not intended to model a useful application. It is a controlled workload for learning how baseline timing and profile evidence relate.

In [3]:
ELEMENT_COUNT = 1_000_000
TRANSFORM_ROUNDS = 3

UINT32_MASK = (1 << 32) - 1
UINT64_MASK = (1 << 64) - 1

input_values = [
    ((index * 2_654_435_761) ^ (index >> 3)) & UINT32_MASK
    for index in range(ELEMENT_COUNT)
]

workload_configuration = {
    "element_count": ELEMENT_COUNT,
    "transform_rounds": TRANSFORM_ROUNDS,
    "histogram_bins": 256,
    "input_size_mib_estimate": (input_values.__sizeof__() / 1024**2),
}

pprint(workload_configuration)

{'element_count': 1000000,
 'histogram_bins': 256,
 'input_size_mib_estimate': 8.057319641113281,
 'transform_rounds': 3}


In [ ]:
def transform_values(
    values: list[int],
    rounds: int,
) -> list[int]:
    transformed = [0] * len(values)

    for index, value in enumerate(values):
        current = value

        for _ in range(rounds):
            current ^= current >> 16
            current = (current * 0x45D9F3B) & UINT32_MASK
            current ^= current >> 16

        transformed[index] = current

    return transformed


def build_low_byte_histogram(values: list[int]) -> list[int]:
    histogram = [0] * 256

    for value in values:
        histogram[value & 0xFF] += 1

    return histogram


def calculate_weighted_checksum(values: list[int]) -> int:
    checksum = 0

    for index, value in enumerate(values, start=1):
        checksum = (checksum + index * (value & 0xFFFF)) & UINT64_MASK

    return checksum


def profiling_workload() -> tuple[int, int, int]:
    transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    histogram = build_low_byte_histogram(transformed)
    checksum = calculate_weighted_checksum(transformed)

    return (
        checksum,
        sum(histogram),
        max(histogram),
    )

### Correctness and determinism check

Before timing the workload, run it twice and verify that:

- every transformed element is represented in the histogram;
- repeated executions produce the same result.

These executions also provide initial interpreter and memory-allocation warm-up, but they are not part of the measured baseline.

In [5]:
with pinned_to_cpu(PROFILE_CPU):
    first_result = profiling_workload()
    second_result = profiling_workload()

assert first_result == second_result
assert first_result[1] == ELEMENT_COUNT
assert 0 < first_result[2] <= ELEMENT_COUNT

correctness_result = {
    "checksum": first_result[0],
    "histogram_total": first_result[1],
    "largest_histogram_bin": first_result[2],
    "deterministic": first_result == second_result,
}

pprint(correctness_result)

{'checksum': 16378977129066911,
 'deterministic': True,
 'histogram_total': 1000000,
 'largest_histogram_bin': 4112}


### Unprofiled baseline

The baseline uses wall-clock time without an active profiler. All warm-up and measured iterations run while the notebook process is pinned to CPU 15.

Individual trials are retained because the distribution matters. A single minimum or mean cannot show scheduling noise, thermal effects, frequency changes, or occasional operating-system interference.

This baseline will later be compared with the runtime of the same workload under profiling.

In [6]:
BASELINE_WARMUP_ITERATIONS = 1
BASELINE_ITERATIONS = 7

with pinned_to_cpu(PROFILE_CPU):
    baseline_result = benchmark_callable(
        profiling_workload,
        warmup_iterations=BASELINE_WARMUP_ITERATIONS,
        iterations=BASELINE_ITERATIONS,
    )

baseline_trials = pd.DataFrame(
    {
        "trial": range(1, baseline_result.iterations + 1),
        "wall_ms": [
            duration_ns / 1_000_000 for duration_ns in baseline_result.durations_ns
        ],
    }
)

baseline_trials.round(3)

,trial,wall_ms
0,1,470.828
1,2,470.727
2,3,470.911
3,4,469.696
4,5,470.071
5,6,468.926
6,7,469.325


In [7]:
baseline_summary = pd.DataFrame(
    [
        {
            "iterations": baseline_result.iterations,
            "minimum_ms": baseline_result.minimum_ns / 1_000_000,
            "median_ms": baseline_result.median_ns / 1_000_000,
            "mean_ms": baseline_result.mean_ns / 1_000_000,
            "maximum_ms": baseline_result.maximum_ns / 1_000_000,
            "standard_deviation_ms": (
                baseline_result.standard_deviation_ns / 1_000_000
            ),
            "relative_standard_deviation_percent": (
                100 * baseline_result.standard_deviation_ns / baseline_result.mean_ns
            ),
        }
    ]
)

baseline_summary.round(3)

,iterations,minimum_ms,median_ms,mean_ms,maximum_ms,standard_deviation_ms,relative_standard_deviation_percent
0,7,468.926,470.071,470.069,470.911,0.729,0.155


### Deterministic function-level profile

`cProfile` is a deterministic profiler: it observes Python call and return events and accumulates timing statistics for functions encountered during execution.

We will collect several independent profiles while retaining the wall-clock duration of each profiled run. This permits two separate comparisons:

1. **profile attribution**: which functions account for the observed profiled runtime;
2. **profiler perturbation**: how much slower the workload becomes while profiling is active.

The profiler is enabled immediately before `profiling_workload()` and disabled immediately afterward. CPU affinity remains fixed at CPU 15.

The profiler's function times are measurements from the instrumented execution. They should not be substituted for the unprofiled baseline.

In [8]:
import cProfile
import pstats
from time import perf_counter_ns


PROFILE_ITERATIONS = 5


def collect_cprofile_trial() -> tuple[
    tuple[int, int, int],
    cProfile.Profile,
    int,
]:
    profiler = cProfile.Profile()

    start_ns = perf_counter_ns()
    profiler.enable()

    try:
        result = profiling_workload()
    finally:
        profiler.disable()
        wall_ns = perf_counter_ns() - start_ns

    return result, profiler, wall_ns

In [9]:
profiled_runs = []

with pinned_to_cpu(PROFILE_CPU):
    for trial in range(1, PROFILE_ITERATIONS + 1):
        result, profiler, wall_ns = collect_cprofile_trial()

        assert result == first_result

        profiled_runs.append(
            {
                "trial": trial,
                "result": result,
                "profiler": profiler,
                "wall_ns": wall_ns,
            }
        )

profiled_trials = pd.DataFrame(
    [
        {
            "trial": run["trial"],
            "wall_ms": run["wall_ns"] / 1_000_000,
        }
        for run in profiled_runs
    ]
)

profiled_trials.round(3)

,trial,wall_ms
0,1,483.402
1,2,490.169
2,3,492.176
3,4,480.899
4,5,492.146


In [10]:
profiled_median_ns = float(profiled_trials["wall_ms"].median()) * 1_000_000
baseline_median_ns = baseline_result.median_ns

profiling_overhead_summary = pd.DataFrame(
    [
        {
            "baseline_median_ms": baseline_median_ns / 1_000_000,
            "profiled_median_ms": profiled_median_ns / 1_000_000,
            "added_median_ms": (profiled_median_ns - baseline_median_ns) / 1_000_000,
            "slowdown_factor": (profiled_median_ns / baseline_median_ns),
            "overhead_percent": (
                100 * (profiled_median_ns - baseline_median_ns) / baseline_median_ns
            ),
        }
    ]
)

profiling_overhead_summary.round(3)

,baseline_median_ms,profiled_median_ms,added_median_ms,slowdown_factor,overhead_percent
0,470.071,490.169,20.097,1.043,4.275


### Selecting a representative profile

The profile whose wall time is closest to the median profiled wall time is used for function-level inspection.

Selecting the median run avoids presenting an unusually fast or slow trial as representative. It does not combine function statistics across runs: the displayed function times all come from one internally consistent execution.

In [11]:
profiled_trials["distance_from_median_ms"] = (
    profiled_trials["wall_ms"] - profiled_trials["wall_ms"].median()
).abs()

representative_trial_index = int(profiled_trials["distance_from_median_ms"].idxmin())
representative_run = profiled_runs[representative_trial_index]

representative_selection = {
    "trial": representative_run["trial"],
    "wall_ms": representative_run["wall_ns"] / 1_000_000,
    "median_profiled_wall_ms": profiled_trials["wall_ms"].median(),
}

pprint(representative_selection)

{'median_profiled_wall_ms': np.float64(490.16875),
 'trial': 2,
 'wall_ms': 490.16875}


### Function-level statistics

For each function, `cProfile` records:

- **primitive calls**: calls not induced by recursion;
- **total calls**: all calls, including recursive calls;
- **internal time** (`tottime`): time spent in the function body, excluding callees;
- **cumulative time** (`cumtime`): time spent in the function and functions it called.

The workload phases do not call one another, so their internal and cumulative times should be similar. `profiling_workload`, however, calls all three phases; its cumulative time should cover nearly the entire instrumented workload while its internal time should remain small.

In [12]:
representative_stats = pstats.Stats(representative_run["profiler"])

profile_rows = []

for function_key, statistics in representative_stats.stats.items():
    filename, line_number, function_name = function_key
    primitive_calls, total_calls, internal_s, cumulative_s, _ = statistics

    profile_rows.append(
        {
            "function": function_name,
            "filename": Path(filename).name,
            "line": line_number,
            "primitive_calls": primitive_calls,
            "total_calls": total_calls,
            "internal_ms": internal_s * 1_000,
            "cumulative_ms": cumulative_s * 1_000,
        }
    )

function_profile = (
    pd.DataFrame(profile_rows)
    .sort_values(
        ["cumulative_ms", "internal_ms"],
        ascending=False,
    )
    .reset_index(drop=True)
)

function_profile.head(15).round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms
0,profiling_workload,1169694572.py,40,1,1,0.013,480.672
1,transform_values,1169694572.py,1,1,1,388.478,388.478
2,calculate_weighted_checksum,1169694572.py,29,1,1,63.785,63.785
3,build_low_byte_histogram,1169694572.py,20,1,1,28.392,28.392
4,<method 'disable' of '_lsprof.Profiler' objects>,~,0,1,1,0.057,0.057
5,<built-in method builtins.sum>,~,0,1,1,0.002,0.002
6,<built-in method builtins.max>,~,0,1,1,0.002,0.002
7,<built-in method builtins.len>,~,0,1,1,0.000,0.000


In [14]:
WORKLOAD_FUNCTIONS = {
    "profiling_workload",
    "transform_values",
    "build_low_byte_histogram",
    "calculate_weighted_checksum",
}

workload_function_profile = (
    function_profile[function_profile["function"].isin(WORKLOAD_FUNCTIONS)]
    .copy()
    .sort_values("cumulative_ms", ascending=False)
    .reset_index(drop=True)
)

representative_wall_ms = representative_run["wall_ns"] / 1_000_000

workload_function_profile["internal_fraction_percent"] = (
    100 * workload_function_profile["internal_ms"] / representative_wall_ms
)

workload_function_profile.round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms,internal_fraction_percent
0,profiling_workload,1169694572.py,40,1,1,0.013,480.672,0.003
1,transform_values,1169694572.py,1,1,1,388.478,388.478,79.254
2,calculate_weighted_checksum,1169694572.py,29,1,1,63.785,63.785,13.013
3,build_low_byte_histogram,1169694572.py,20,1,1,28.392,28.392,5.792


### Minimally instrumented phase timing

The deterministic profile attributed most runtime to `transform_values`, followed by the checksum and histogram phases. Those measurements were collected while `cProfile` was active.

To test whether profiling materially changes the relative phase distribution, the workload is repeated with explicit wall-clock timestamps around the three phase boundaries.

This instrumentation adds only a small fixed number of timer calls per workload invocation. It still perturbs execution and does not expose activity inside each phase, but it provides an independent timing method with substantially less instrumentation than deterministic call profiling.

In [45]:
PHASE_TIMING_ITERATIONS = 7


def timed_profiling_workload() -> dict[str, int]:
    workload_start_ns = perf_counter_ns()

    transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    transform_end_ns = perf_counter_ns()

    histogram = build_low_byte_histogram(transformed)
    histogram_end_ns = perf_counter_ns()

    checksum = calculate_weighted_checksum(transformed)
    checksum_end_ns = perf_counter_ns()

    result = (
        checksum,
        sum(histogram),
        max(histogram),
    )
    result_end_ns = perf_counter_ns()

    del transformed
    del histogram
    cleanup_end_ns = perf_counter_ns()

    assert result == first_result

    return {
        "transform_ns": transform_end_ns - workload_start_ns,
        "histogram_ns": histogram_end_ns - transform_end_ns,
        "checksum_ns": checksum_end_ns - histogram_end_ns,
        "finalization_ns": result_end_ns - checksum_end_ns,
        "cleanup_ns": cleanup_end_ns - result_end_ns,
        "workload_ns": cleanup_end_ns - workload_start_ns,
    }

In [46]:
phase_measurements = []

with pinned_to_cpu(PROFILE_CPU):
    timed_profiling_workload()

    for trial in range(1, PHASE_TIMING_ITERATIONS + 1):
        measurement = timed_profiling_workload()
        measurement["trial"] = trial
        phase_measurements.append(measurement)

phase_trials = pd.DataFrame(phase_measurements)

for column in (
    "transform_ns",
    "histogram_ns",
    "checksum_ns",
    "finalization_ns",
    "cleanup_ns",
    "workload_ns",
):
    phase_trials[column.removesuffix("_ns") + "_ms"] = phase_trials[column] / 1_000_000

phase_trials[
    [
        "trial",
        "transform_ms",
        "histogram_ms",
        "checksum_ms",
        "finalization_ms",
        "cleanup_ms",
        "workload_ms",
    ]
].round(3)

,trial,transform_ms,histogram_ms,checksum_ms,finalization_ms,cleanup_ms,workload_ms
0,1,355.175,30.827,62.882,0.005,6.414,455.302
1,2,355.211,30.704,62.704,0.004,6.144,454.767
2,3,355.265,29.011,63.053,0.004,5.916,453.249
3,4,354.261,30.668,64.442,0.004,5.839,455.214
4,5,354.142,30.664,62.974,0.004,5.608,453.392
5,6,351.080,30.792,62.261,0.005,5.597,449.735
6,7,347.095,31.282,60.573,0.005,5.531,444.485


In [47]:
phase_summary = pd.DataFrame(
    [
        {
            "phase": "transform_values",
            "median_ms": phase_trials["transform_ms"].median(),
        },
        {
            "phase": "build_low_byte_histogram",
            "median_ms": phase_trials["histogram_ms"].median(),
        },
        {
            "phase": "calculate_weighted_checksum",
            "median_ms": phase_trials["checksum_ms"].median(),
        },
        {
            "phase": "result_finalization",
            "median_ms": phase_trials["finalization_ms"].median(),
        },
        {
            "phase": "local_cleanup",
            "median_ms": phase_trials["cleanup_ms"].median(),
        },
    ]
)

median_phase_total_ms = phase_summary["median_ms"].sum()

phase_summary["fraction_percent"] = (
    100 * phase_summary["median_ms"] / median_phase_total_ms
)

phase_summary.round(3)

,phase,median_ms,fraction_percent
0,transform_values,354.261,78.084
1,build_low_byte_histogram,30.704,6.768
2,calculate_weighted_checksum,62.882,13.860
3,result_finalization,0.004,0.001
4,local_cleanup,5.839,1.287


In [48]:
MANUAL_ONLY_PHASES = {
    "result_finalization",
    "local_cleanup",
}

cprofile_phase_comparison = (
    workload_function_profile[
        workload_function_profile["function"] != "profiling_workload"
    ][
        [
            "function",
            "internal_ms",
            "internal_fraction_percent",
        ]
    ]
    .rename(
        columns={
            "internal_ms": "cprofile_internal_ms",
            "internal_fraction_percent": "cprofile_fraction_percent",
        }
    )
    .merge(
        phase_summary[~phase_summary["phase"].isin(MANUAL_ONLY_PHASES)].rename(
            columns={"phase": "function"}
        ),
        on="function",
        how="inner",
    )
)

cprofile_phase_comparison["cprofile_to_manual_time_ratio"] = (
    cprofile_phase_comparison["cprofile_internal_ms"]
    / cprofile_phase_comparison["median_ms"]
)

cprofile_phase_comparison["fraction_difference_percentage_points"] = (
    cprofile_phase_comparison["cprofile_fraction_percent"]
    - cprofile_phase_comparison["fraction_percent"]
)

cprofile_phase_comparison.round(3)

,function,cprofile_internal_ms,cprofile_fraction_percent,median_ms,fraction_percent,cprofile_to_manual_time_ratio,fraction_difference_percentage_points
0,transform_values,388.478,79.254,354.261,78.084,1.097,1.169
1,calculate_weighted_checksum,63.785,13.013,62.882,13.860,1.014,-0.847
2,build_low_byte_histogram,28.392,5.792,30.704,6.768,0.925,-0.975


In [49]:
phase_timing_overhead = pd.DataFrame(
    [
        {
            "baseline_median_ms": baseline_result.median_ns / 1_000_000,
            "phase_timed_median_ms": phase_trials["workload_ms"].median(),
            "slowdown_factor": (
                phase_trials["workload_ms"].median()
                / (baseline_result.median_ns / 1_000_000)
            ),
            "overhead_percent": (
                100
                * (
                    phase_trials["workload_ms"].median()
                    - baseline_result.median_ns / 1_000_000
                )
                / (baseline_result.median_ns / 1_000_000)
            ),
        }
    ]
)

phase_timing_overhead.round(3)

,baseline_median_ms,phase_timed_median_ms,slowdown_factor,overhead_percent
0,470.071,453.392,0.965,-3.548


### Interleaved measurement of instrumentation overhead

The earlier baseline and phase-timed trials were collected in separate blocks. Their runtime difference may therefore include temporal drift rather than timer overhead alone.

To reduce this confounding effect, the next experiment alternates uninstrumented and phase-timed executions. Odd-numbered trial pairs run one order; even-numbered pairs reverse it:

- odd pair: uninstrumented, then phase timed;
- even pair: phase timed, then uninstrumented.

Alternating order distributes short-term drift and order effects across both configurations. Each execution is still an independent workload invocation, and all trials remain pinned to CPU 15.

The paired difference within each trial pair is the primary overhead measurement.

In [50]:
INTERLEAVED_PAIRS = 10


def measure_uninstrumented_workload() -> dict[str, int | str]:
    start_ns = perf_counter_ns()
    result = profiling_workload()
    end_ns = perf_counter_ns()

    assert result == first_result

    return {
        "configuration": "uninstrumented",
        "wall_ns": end_ns - start_ns,
    }


def measure_phase_timed_workload() -> dict[str, int | str]:
    measurement = timed_profiling_workload()

    return {
        "configuration": "phase timed",
        "wall_ns": measurement["workload_ns"],
    }

In [51]:
interleaved_measurements = []

with pinned_to_cpu(PROFILE_CPU):
    measure_uninstrumented_workload()
    measure_phase_timed_workload()

    for pair in range(1, INTERLEAVED_PAIRS + 1):
        if pair % 2 == 1:
            ordered_measurements = (
                measure_uninstrumented_workload(),
                measure_phase_timed_workload(),
            )
        else:
            ordered_measurements = (
                measure_phase_timed_workload(),
                measure_uninstrumented_workload(),
            )

        for order, measurement in enumerate(
            ordered_measurements,
            start=1,
        ):
            interleaved_measurements.append(
                {
                    "pair": pair,
                    "order": order,
                    **measurement,
                }
            )

interleaved_trials = pd.DataFrame(interleaved_measurements)
interleaved_trials["wall_ms"] = interleaved_trials["wall_ns"] / 1_000_000

interleaved_trials[
    [
        "pair",
        "order",
        "configuration",
        "wall_ms",
    ]
].round(3)

,pair,order,configuration,wall_ms
0,1,1,uninstrumented,459.446
1,1,2,phase timed,458.254
2,2,1,phase timed,459.273
3,2,2,uninstrumented,457.859
4,3,1,uninstrumented,459.301
5,3,2,phase timed,456.996
6,4,1,phase timed,456.399
7,4,2,uninstrumented,457.950
8,5,1,uninstrumented,457.549
9,5,2,phase timed,459.928


In [52]:
paired_overhead = interleaved_trials.pivot(
    index="pair",
    columns="configuration",
    values="wall_ms",
).reset_index()

paired_overhead["added_ms"] = (
    paired_overhead["phase timed"] - paired_overhead["uninstrumented"]
)

paired_overhead["overhead_percent"] = (
    100 * paired_overhead["added_ms"] / paired_overhead["uninstrumented"]
)

paired_overhead.round(3)

configuration,pair,phase timed,uninstrumented,added_ms,overhead_percent
0,1,458.254,459.446,-1.193,-0.260
1,2,459.273,457.859,1.414,0.309
2,3,456.996,459.301,-2.305,-0.502
3,4,456.399,457.950,-1.551,-0.339
4,5,459.928,457.549,2.379,0.520
5,6,454.589,457.114,-2.525,-0.552
6,7,455.562,455.278,0.283,0.062
7,8,453.349,457.725,-4.377,-0.956
8,9,452.021,456.586,-4.565,-1.000
9,10,454.401,455.401,-1.000,-0.220


In [53]:
interleaved_summary = pd.DataFrame(
    [
        {
            "uninstrumented_median_ms": (paired_overhead["uninstrumented"].median()),
            "phase_timed_median_ms": (paired_overhead["phase timed"].median()),
            "median_paired_added_ms": (paired_overhead["added_ms"].median()),
            "median_paired_overhead_percent": (
                paired_overhead["overhead_percent"].median()
            ),
            "minimum_paired_overhead_percent": (
                paired_overhead["overhead_percent"].min()
            ),
            "maximum_paired_overhead_percent": (
                paired_overhead["overhead_percent"].max()
            ),
        }
    ]
)

interleaved_summary.round(3)

,uninstrumented_median_ms,phase_timed_median_ms,median_paired_added_ms,median_paired_overhead_percent,minimum_paired_overhead_percent,maximum_paired_overhead_percent
0,457.637,455.98,-1.372,-0.299,-1.0,0.52


In [54]:
order_effect_summary = (
    interleaved_trials.groupby(
        ["configuration", "order"],
        as_index=False,
    )["wall_ms"]
    .agg(["median", "mean", "std"])
    .reset_index()
)

order_effect_summary.round(3)

,index,configuration,order,median,mean,std
0,0,phase timed,1,454.589,455.602,2.327
1,1,phase timed,2,456.996,456.552,3.000
2,2,uninstrumented,1,457.549,457.632,1.783
3,3,uninstrumented,2,457.725,457.210,1.063


## Observations

The controlled workload was deterministic and produced the same checksum and histogram summary on repeated executions.

The initial unprofiled baseline had a median wall time of 470.071 ms across seven trials. Its relative standard deviation was 0.155%, providing a stable reference at that point in the notebook execution.

Running the workload under `cProfile` produced a median wall time of 490.169 ms. Relative to the initial baseline, this was:

- 20.097 ms additional median runtime;
- a 1.043× slowdown;
- 4.275% apparent profiling overhead.

In the representative `cProfile` run, internal time was attributed primarily to:

- `transform_values`: 388.478 ms;
- `calculate_weighted_checksum`: 63.785 ms;
- `build_low_byte_histogram`: 28.392 ms.

`profiling_workload` itself had negligible internal time but 480.672 ms cumulative time because it called all three phases.

Independent manual phase timing reproduced the same ordering. After including result construction and local-object cleanup, the median phase distribution was:

- transformation: 354.261 ms, or 78.084%;
- checksum: 62.882 ms, or 13.860%;
- histogram: 30.704 ms, or 6.768%;
- local cleanup: 5.839 ms, or 1.287%;
- result finalization: 0.004 ms, or 0.001%.

An early comparison incorrectly suggested that manual instrumentation made the workload faster. Interleaving the configurations did not initially remove this difference because their timing boundaries were unequal: the manually timed path stopped before releasing its million-element transformed list.

After explicitly including local cleanup, the interleaved comparison produced:

- uninstrumented median: 457.637 ms;
- phase-timed median: 455.980 ms;
- median paired difference: -1.372 ms, or -0.299%;
- paired differences ranging from -1.000% to +0.520%.

Because the corrected paired differences cross zero and are small relative to runtime variation, the overhead of the manual timestamps was not resolvable in this experiment.

## Explanation

The prediction that transformation would dominate was supported by two independent measurement methods.

`cProfile` identifies expensive Python call paths. Its distinction between internal and cumulative time was important:

- `profiling_workload` had high cumulative time because it contained the complete call path;
- `transform_values` had the greatest internal time because its own body performed most of the work.

The profiler does not explain why `transform_values` was expensive at the processor level. Its output cannot distinguish Python bytecode dispatch, arbitrary-precision integer operations, allocation, cache behavior, branch behavior, or instruction throughput.

The approximately 4.3% `cProfile` slowdown is specific to this workload and profiler configuration. This workload makes only a few long-running Python function calls. A call-heavy workload could experience substantially greater deterministic-profiling overhead.

Manual phase timing produced almost the same relative attribution with much less instrumentation. However, the sequence of timing experiments showed that measurement boundaries must include semantically identical work.

In particular, an internal timestamp recorded before a function returns can omit reference-count decrements and deallocation of local objects. Releasing the transformed list of one million Python integers took approximately 5.8 ms. That work was included naturally in the externally timed function call but was initially excluded from the internal phase timer.

After matching the boundaries and interleaving trial order, the remaining manual-instrumentation effect was smaller than the observed noise. The experiment therefore supports only an upper-bound-style conclusion: manual timestamp overhead was negligible relative to a roughly 456 ms workload, not that it was zero.

The profiling evidence supports optimization of `transform_values` first. It does not yet establish which implementation change would be effective.

## Optimization hypothesis

Both `cProfile` and manual phase timing identified `transform_values` as the dominant phase, accounting for approximately 78–79% of total runtime.

The transformation applies the same independent sequence of 32-bit arithmetic operations to every input element. This is a suitable array operation: NumPy can execute the elementwise loop in compiled code rather than dispatching Python bytecode for every element and every transformation round.

The optimized implementation will change only the transformation phase:

- the input values will be represented as a preconstructed `numpy.uint32` array;
- each transformation round will use NumPy bitwise and multiplication operations;
- the transformed array will be converted back to a Python list;
- the existing Python histogram and checksum functions will remain unchanged.

The conversion back to a list is deliberately included in the optimized transformation time. It is required by the unchanged downstream functions and therefore represents a real integration cost.

Prediction:

1. the NumPy transformation should be substantially faster than the Python transformation;
2. the end-to-end speedup will be smaller than the transformation-only speedup because histogram construction, checksum calculation, conversion to a list, and cleanup remain;
3. the optimized output should be bit-for-bit equivalent to the original transformation.

In [63]:
import numpy as np


numpy_input_values = np.asarray(input_values, dtype=np.uint32)

numpy_optimization_environment = {
    "numpy_version": np.__version__,
    "input_dtype": str(numpy_input_values.dtype),
    "input_shape": numpy_input_values.shape,
    "input_nbytes_mib": numpy_input_values.nbytes / 1024**2,
}

pprint(numpy_optimization_environment)

{'input_dtype': 'uint32',
 'input_nbytes_mib': 3.814697265625,
 'input_shape': (1000000,),
 'numpy_version': '2.5.1'}


In [56]:
def transform_values_numpy(
    values: np.ndarray,
    rounds: int,
) -> list[int]:
    current = values.copy()

    for _ in range(rounds):
        current ^= current >> np.uint32(16)
        current *= np.uint32(0x45D9F3B)
        current ^= current >> np.uint32(16)

    return current.tolist()

### Equivalence validation

The optimized transformation must produce exactly the same transformed integers as the original Python implementation.

The validation compares the complete one-million-element result, not merely the final checksum. It also runs the unchanged histogram and checksum phases over the optimized output and verifies the complete workload result.

In [64]:
with pinned_to_cpu(PROFILE_CPU):
    reference_transformed = transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    numpy_transformed = transform_values_numpy(
        numpy_input_values,
        TRANSFORM_ROUNDS,
    )

assert numpy_transformed == reference_transformed

numpy_histogram = build_low_byte_histogram(numpy_transformed)
numpy_checksum = calculate_weighted_checksum(numpy_transformed)

numpy_validation_result = (
    numpy_checksum,
    sum(numpy_histogram),
    max(numpy_histogram),
)

assert numpy_validation_result == first_result

optimization_validation = {
    "elementwise_equal": numpy_transformed == reference_transformed,
    "element_count": len(numpy_transformed),
    "workload_result_equal": numpy_validation_result == first_result,
    "checksum": numpy_validation_result[0],
}

pprint(optimization_validation)

del reference_transformed
del numpy_transformed
del numpy_histogram

{'checksum': 16378977129066911,
 'element_count': 1000000,
 'elementwise_equal': True,
 'workload_result_equal': True}


In [65]:
def optimized_profiling_workload() -> tuple[int, int, int]:
    transformed = transform_values_numpy(
        numpy_input_values,
        TRANSFORM_ROUNDS,
    )
    histogram = build_low_byte_histogram(transformed)
    checksum = calculate_weighted_checksum(transformed)

    return (
        checksum,
        sum(histogram),
        max(histogram),
    )

### Interleaved before/after measurement

The original and optimized workloads are measured in alternating order.

Odd-numbered pairs run the original first; even-numbered pairs run the optimized version first. This distributes short-term drift and order effects across both implementations.

The primary end-to-end metric is the paired speedup:

\[
\text{speedup} =
\frac{\text{original wall time}}
     {\text{optimized wall time}}
\]

A value greater than one means that the optimized workload is faster.

In [66]:
OPTIMIZATION_PAIRS = 10


def measure_workload_variant(
    name: str,
    function,
) -> dict[str, int | str]:
    start_ns = perf_counter_ns()
    result = function()
    end_ns = perf_counter_ns()

    assert result == first_result

    return {
        "implementation": name,
        "wall_ns": end_ns - start_ns,
    }

In [ ]:
optimization_measurements = []

with pinned_to_cpu(PROFILE_CPU):
    profiling_workload()
    optimized_profiling_workload()

    for pair in range(1, OPTIMIZATION_PAIRS + 1):
        if pair % 2 == 1:
            ordered_measurements = (
                measure_workload_variant(
                    "original Python",
                    profiling_workload,
                ),
                measure_workload_variant(
                    "NumPy transform",
                    optimized_profiling_workload,
                ),
            )
        else:
            ordered_measurements = (
                measure_workload_variant(
                    "NumPy transform",
                    optimized_profiling_workload,
                ),
                measure_workload_variant(
                    "original Python",
                    profiling_workload,
                ),
            )

        for order, measurement in enumerate(
            ordered_measurements,
            start=1,
        ):
            optimization_measurements.append(
                {
                    "pair": pair,
                    "order": order,
                    **measurement,
                }
            )

optimization_trials = pd.DataFrame(optimization_measurements)
optimization_trials["wall_ms"] = optimization_trials["wall_ns"] / 1_000_000

optimization_trials[
    [
        "pair",
        "order",
        "implementation",
        "wall_ms",
    ]
].round(3)

,pair,order,implementation,wall_ms
0,1,1,original Python,446.397
1,1,2,NumPy transform,106.047
2,2,1,NumPy transform,107.245
3,2,2,original Python,448.041
4,3,1,original Python,452.326
5,3,2,NumPy transform,107.491
6,4,1,NumPy transform,108.015
7,4,2,original Python,445.207
8,5,1,original Python,447.908
9,5,2,NumPy transform,107.351


In [ ]:
paired_optimization = optimization_trials.pivot(
    index="pair",
    columns="implementation",
    values="wall_ms",
).reset_index()

paired_optimization["saved_ms"] = (
    paired_optimization["original Python"] - paired_optimization["NumPy transform"]
)

paired_optimization["speedup"] = (
    paired_optimization["original Python"] / paired_optimization["NumPy transform"]
)

paired_optimization["runtime_reduction_percent"] = (
    100 * paired_optimization["saved_ms"] / paired_optimization["original Python"]
)

paired_optimization.round(3)

implementation,pair,NumPy transform,original Python,saved_ms,speedup,runtime_reduction_percent
0,1,106.047,446.397,340.350,4.209,76.244
1,2,107.245,448.041,340.796,4.178,76.064
2,3,107.491,452.326,344.835,4.208,76.236
3,4,108.015,445.207,337.192,4.122,75.738
4,5,107.351,447.908,340.557,4.172,76.033
5,6,108.307,450.255,341.948,4.157,75.945
6,7,107.152,449.624,342.471,4.196,76.169
7,8,105.220,446.892,341.672,4.247,76.455
8,9,107.581,447.166,339.585,4.157,75.942
9,10,106.788,448.099,341.311,4.196,76.169


In [ ]:
optimization_summary = pd.DataFrame(
    [
        {
            "original_median_ms": (paired_optimization["original Python"].median()),
            "optimized_median_ms": (paired_optimization["NumPy transform"].median()),
            "median_saved_ms": (paired_optimization["saved_ms"].median()),
            "median_speedup": (paired_optimization["speedup"].median()),
            "median_runtime_reduction_percent": (
                paired_optimization["runtime_reduction_percent"].median()
            ),
            "minimum_speedup": (paired_optimization["speedup"].min()),
            "maximum_speedup": (paired_optimization["speedup"].max()),
        }
    ]
)

optimization_summary.round(3)

,original_median_ms,optimized_median_ms,median_saved_ms,median_speedup,median_runtime_reduction_percent,minimum_speedup,maximum_speedup
0,447.975,107.298,341.054,4.187,76.116,4.122,4.247


### Transformation-only measurement

The end-to-end comparison includes the unchanged histogram and checksum phases. To isolate the optimization itself, the two transformation implementations are also measured directly.

Both functions return a Python list. The NumPy measurement therefore includes copying the input array, executing the vectorized operations, and converting the result to Python integers.

In [70]:
TRANSFORM_OPTIMIZATION_PAIRS = 10


def measure_transform_variant(
    name: str,
    function,
) -> dict[str, int | str]:
    start_ns = perf_counter_ns()
    transformed = function()
    end_ns = perf_counter_ns()

    assert len(transformed) == ELEMENT_COUNT
    assert transformed[0] == reference_first_value
    assert transformed[-1] == reference_last_value

    return {
        "implementation": name,
        "wall_ns": end_ns - start_ns,
    }


reference_first_value = transform_values(
    input_values[:1],
    TRANSFORM_ROUNDS,
)[0]

reference_last_value = transform_values(
    input_values[-1:],
    TRANSFORM_ROUNDS,
)[0]

In [ ]:
transform_optimization_measurements = []

with pinned_to_cpu(PROFILE_CPU):
    transform_values(
        input_values,
        TRANSFORM_ROUNDS,
    )
    transform_values_numpy(
        numpy_input_values,
        TRANSFORM_ROUNDS,
    )

    for pair in range(1, TRANSFORM_OPTIMIZATION_PAIRS + 1):
        if pair % 2 == 1:
            ordered_measurements = (
                measure_transform_variant(
                    "original Python",
                    lambda: transform_values(
                        input_values,
                        TRANSFORM_ROUNDS,
                    ),
                ),
                measure_transform_variant(
                    "NumPy",
                    lambda: transform_values_numpy(
                        numpy_input_values,
                        TRANSFORM_ROUNDS,
                    ),
                ),
            )
        else:
            ordered_measurements = (
                measure_transform_variant(
                    "NumPy",
                    lambda: transform_values_numpy(
                        numpy_input_values,
                        TRANSFORM_ROUNDS,
                    ),
                ),
                measure_transform_variant(
                    "original Python",
                    lambda: transform_values(
                        input_values,
                        TRANSFORM_ROUNDS,
                    ),
                ),
            )

        for order, measurement in enumerate(
            ordered_measurements,
            start=1,
        ):
            transform_optimization_measurements.append(
                {
                    "pair": pair,
                    "order": order,
                    **measurement,
                }
            )

transform_optimization_trials = pd.DataFrame(transform_optimization_measurements)
transform_optimization_trials["wall_ms"] = (
    transform_optimization_trials["wall_ns"] / 1_000_000
)

transform_optimization_trials[
    [
        "pair",
        "order",
        "implementation",
        "wall_ms",
    ]
].round(3)

,pair,order,implementation,wall_ms
0,1,1,original Python,349.256
1,1,2,NumPy,8.851
2,2,1,NumPy,9.049
3,2,2,original Python,349.567
4,3,1,original Python,349.811
5,3,2,NumPy,9.137
6,4,1,NumPy,9.154
7,4,2,original Python,352.864
8,5,1,original Python,353.011
9,5,2,NumPy,9.045


In [ ]:
paired_transform_optimization = transform_optimization_trials.pivot(
    index="pair",
    columns="implementation",
    values="wall_ms",
).reset_index()

paired_transform_optimization["saved_ms"] = (
    paired_transform_optimization["original Python"]
    - paired_transform_optimization["NumPy"]
)

paired_transform_optimization["speedup"] = (
    paired_transform_optimization["original Python"]
    / paired_transform_optimization["NumPy"]
)

paired_transform_optimization["runtime_reduction_percent"] = (
    100
    * paired_transform_optimization["saved_ms"]
    / paired_transform_optimization["original Python"]
)

paired_transform_optimization.round(3)

implementation,pair,NumPy,original Python,saved_ms,speedup,runtime_reduction_percent
0,1,8.851,349.256,340.405,39.458,97.466
1,2,9.049,349.567,340.518,38.630,97.411
2,3,9.137,349.811,340.674,38.286,97.388
3,4,9.154,352.864,343.710,38.548,97.406
4,5,9.045,353.011,343.966,39.028,97.438
5,6,8.987,351.081,342.094,39.067,97.440
6,7,8.989,353.462,344.473,39.320,97.457
7,8,9.305,353.823,344.518,38.025,97.370
8,9,9.038,354.740,345.702,39.249,97.452
9,10,9.150,354.814,345.663,38.777,97.421


In [ ]:
transform_optimization_summary = pd.DataFrame(
    [
        {
            "python_median_ms": (
                paired_transform_optimization["original Python"].median()
            ),
            "numpy_median_ms": (paired_transform_optimization["NumPy"].median()),
            "median_saved_ms": (paired_transform_optimization["saved_ms"].median()),
            "median_speedup": (paired_transform_optimization["speedup"].median()),
            "median_runtime_reduction_percent": (
                paired_transform_optimization["runtime_reduction_percent"].median()
            ),
        }
    ]
)

transform_optimization_summary.round(3)

,python_median_ms,numpy_median_ms,median_saved_ms,median_speedup,median_runtime_reduction_percent
0,352.938,9.047,343.838,38.902,97.429


### Optimization results

The NumPy transformation produced an element-for-element identical result for all one million input values. The final checksum and histogram summary were also unchanged.

For the transformation phase alone:

- original Python median: 352.938 ms;
- NumPy median, including conversion back to a Python list: 9.047 ms;
- median time saved: 343.838 ms;
- median speedup: 38.902×;
- median runtime reduction: 97.429%.

For the complete workload:

- original median: 447.975 ms;
- optimized median: 107.298 ms;
- median time saved: 341.054 ms;
- median speedup: 4.187×;
- median runtime reduction: 76.116%.

The ten paired end-to-end speedups ranged from 4.122× to 4.247×.

### Optimization interpretation

The optimization hypothesis was supported. Moving the independent 32-bit transformation loop from Python into NumPy reduced the transformation phase by approximately 97.4%.

The transformation-only speedup was much larger than the end-to-end speedup because the histogram and checksum phases remained implemented as Python loops. Once the dominant transformation phase was reduced from roughly 353 ms to 9 ms, those previously secondary phases became the principal contributors to total runtime.

This is an example of bottleneck migration. Profiling identified the original bottleneck, but after optimizing it, the old profile no longer describes the optimized program. Further optimization decisions require profiling the new implementation rather than continuing to act on the original evidence.

The experiment supports the inference that compiled array execution eliminates most per-element Python interpreter and Python-integer overhead. It does not establish the exact SIMD instructions, cache behavior, or processor-level limiting factor inside the NumPy implementation.

### Re-profile after optimization

Optimization changes the runtime distribution. The original profile cannot be used to identify the bottleneck in the optimized workload.

The optimized workload is therefore profiled independently with `cProfile`. The expected result is that:

- `transform_values_numpy` will no longer dominate;
- the Python checksum and histogram functions will account for most internal time;
- `tolist`, NumPy operations, or array-copy operations may appear as compiled-method calls;
- deterministic-profiling overhead may differ because the optimized workload is much shorter.

In [74]:
OPTIMIZED_PROFILE_ITERATIONS = 5

optimized_profiled_runs = []

with pinned_to_cpu(PROFILE_CPU):
    for trial in range(1, OPTIMIZED_PROFILE_ITERATIONS + 1):
        profiler = cProfile.Profile()

        start_ns = perf_counter_ns()
        profiler.enable()

        try:
            result = optimized_profiling_workload()
        finally:
            profiler.disable()
            wall_ns = perf_counter_ns() - start_ns

        assert result == first_result

        optimized_profiled_runs.append(
            {
                "trial": trial,
                "profiler": profiler,
                "wall_ns": wall_ns,
            }
        )

optimized_profiled_trials = pd.DataFrame(
    [
        {
            "trial": run["trial"],
            "wall_ms": run["wall_ns"] / 1_000_000,
        }
        for run in optimized_profiled_runs
    ]
)

optimized_profiled_trials.round(3)

,trial,wall_ms
0,1,110.495
1,2,110.197
2,3,108.800
3,4,108.975
4,5,110.555


In [ ]:
optimized_profiled_trials["distance_from_median_ms"] = (
    optimized_profiled_trials["wall_ms"] - optimized_profiled_trials["wall_ms"].median()
).abs()

optimized_representative_index = int(
    optimized_profiled_trials["distance_from_median_ms"].idxmin()
)

optimized_representative_run = optimized_profiled_runs[optimized_representative_index]

optimized_profile_selection = {
    "trial": optimized_representative_run["trial"],
    "wall_ms": (optimized_representative_run["wall_ns"] / 1_000_000),
    "median_profiled_wall_ms": (optimized_profiled_trials["wall_ms"].median()),
}

pprint(optimized_profile_selection)

{'median_profiled_wall_ms': np.float64(110.196998),
 'trial': 2,
 'wall_ms': 110.196998}


In [ ]:
optimized_stats = pstats.Stats(optimized_representative_run["profiler"])

optimized_profile_rows = []

for function_key, statistics in optimized_stats.stats.items():
    filename, line_number, function_name = function_key
    primitive_calls, total_calls, internal_s, cumulative_s, _ = statistics

    optimized_profile_rows.append(
        {
            "function": function_name,
            "filename": Path(filename).name,
            "line": line_number,
            "primitive_calls": primitive_calls,
            "total_calls": total_calls,
            "internal_ms": internal_s * 1_000,
            "cumulative_ms": cumulative_s * 1_000,
        }
    )

optimized_function_profile = (
    pd.DataFrame(optimized_profile_rows)
    .sort_values(
        ["cumulative_ms", "internal_ms"],
        ascending=False,
    )
    .reset_index(drop=True)
)

optimized_function_profile.head(15).round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms
0,optimized_profiling_workload,1006304620.py,1,1,1,0.016,104.243
1,calculate_weighted_checksum,1169694572.py,29,1,1,63.302,63.302
2,build_low_byte_histogram,1169694572.py,20,1,1,31.667,31.667
3,transform_values_numpy,729747705.py,1,1,1,1.431,9.254
4,<method 'tolist' of 'numpy.ndarray' objects>,~,0,1,1,7.659,7.659
5,<method 'copy' of 'numpy.ndarray' objects>,~,0,1,1,0.164,0.164
6,<method 'disable' of '_lsprof.Profiler' objects>,~,0,1,1,0.060,0.060
7,<built-in method builtins.max>,~,0,1,1,0.002,0.002
8,<built-in method builtins.sum>,~,0,1,1,0.002,0.002


In [ ]:
OPTIMIZED_WORKLOAD_FUNCTIONS = {
    "optimized_profiling_workload",
    "transform_values_numpy",
    "build_low_byte_histogram",
    "calculate_weighted_checksum",
}

optimized_workload_profile = (
    optimized_function_profile[
        optimized_function_profile["function"].isin(OPTIMIZED_WORKLOAD_FUNCTIONS)
    ]
    .copy()
    .sort_values("cumulative_ms", ascending=False)
    .reset_index(drop=True)
)

optimized_wall_ms = optimized_representative_run["wall_ns"] / 1_000_000

optimized_workload_profile["internal_fraction_percent"] = (
    100 * optimized_workload_profile["internal_ms"] / optimized_wall_ms
)

optimized_workload_profile.round(3)

,function,filename,line,primitive_calls,total_calls,internal_ms,cumulative_ms,internal_fraction_percent
0,optimized_profiling_workload,1006304620.py,1,1,1,0.016,104.243,0.015
1,calculate_weighted_checksum,1169694572.py,29,1,1,63.302,63.302,57.445
2,build_low_byte_histogram,1169694572.py,20,1,1,31.667,31.667,28.737
3,transform_values_numpy,729747705.py,1,1,1,1.431,9.254,1.298


In [ ]:
before_after_profile = workload_function_profile[
    workload_function_profile["function"].isin(
        {
            "transform_values",
            "build_low_byte_histogram",
            "calculate_weighted_checksum",
        }
    )
][
    [
        "function",
        "internal_ms",
        "internal_fraction_percent",
    ]
].rename(
    columns={
        "function": "original_function",
        "internal_ms": "original_internal_ms",
        "internal_fraction_percent": ("original_internal_fraction_percent"),
    }
)

optimized_phase_names = optimized_workload_profile[
    optimized_workload_profile["function"] != "optimized_profiling_workload"
][
    [
        "function",
        "internal_ms",
        "internal_fraction_percent",
    ]
].rename(
    columns={
        "function": "optimized_function",
        "internal_ms": "optimized_internal_ms",
        "internal_fraction_percent": ("optimized_internal_fraction_percent"),
    }
)

profile_phase_mapping = pd.DataFrame(
    [
        {
            "phase": "transformation",
            "original_function": "transform_values",
            "optimized_function": "transform_values_numpy",
        },
        {
            "phase": "histogram",
            "original_function": "build_low_byte_histogram",
            "optimized_function": "build_low_byte_histogram",
        },
        {
            "phase": "checksum",
            "original_function": "calculate_weighted_checksum",
            "optimized_function": "calculate_weighted_checksum",
        },
    ]
)

before_after_profile = profile_phase_mapping.merge(
    before_after_profile,
    on="original_function",
    how="left",
).merge(
    optimized_phase_names,
    on="optimized_function",
    how="left",
)

before_after_profile.round(3)

,phase,original_function,optimized_function,original_internal_ms,original_internal_fraction_percent,optimized_internal_ms,optimized_internal_fraction_percent
0,transformation,transform_values,transform_values_numpy,388.478,79.254,1.431,1.298
1,histogram,build_low_byte_histogram,build_low_byte_histogram,28.392,5.792,31.667,28.737
2,checksum,calculate_weighted_checksum,calculate_weighted_checksum,63.785,13.013,63.302,57.445


### Re-profiling after optimization

The optimized workload had a median profiled wall time of 110.197 ms. The representative run attributed internal time primarily to:

- `calculate_weighted_checksum`: 63.302 ms, or 57.445% of profiled wall time;
- `build_low_byte_histogram`: 31.667 ms, or 28.737%;
- `transform_values_numpy`: 1.431 ms, or 1.298%.

`transform_values_numpy` had 9.254 ms cumulative time. A compiled method called by it accounted for 7.659 ms, showing that much of the remaining transformation-path cost was below the Python function body.

Before optimization, transformation accounted for 388.478 ms and 79.254% of the representative profile. After optimization, its Python function body accounted for only 1.431 ms and 1.298%.

The checksum remained nearly unchanged in absolute time:

- before: 63.785 ms;
- after: 63.302 ms.

Histogram time was also of the same order:

- before: 28.392 ms;
- after: 31.667 ms.

Their much larger optimized fractions therefore reflect bottleneck migration rather than a substantial regression in either function.

### Bottleneck migration after optimization

Re-profiling confirmed that the original profile no longer described the optimized workload.

The weighted checksum became the largest Python bottleneck, accounting for approximately 57.4% of profiled wall time. Histogram construction became the second-largest bottleneck at approximately 28.7%. Their absolute durations remained similar to the original profile; their relative importance increased because the transformation phase had been reduced by almost two orders of magnitude.

The optimized transformation also demonstrates the boundary of deterministic Python profiling. `transform_values_numpy` had little internal Python time but greater cumulative time because it delegated work to compiled NumPy operations. Function-level profiling can attribute this time to the transformation call path, but it cannot reveal the NumPy loop's instruction mix, SIMD utilization, cache behavior, or memory bandwidth.

Optimization must therefore be iterative:

1. measure the original program;
2. optimize the evidenced bottleneck;
3. validate correctness and end-to-end improvement;
4. discard the old bottleneck ranking;
5. profile the changed program again.

The optimized profile now points to the checksum first and histogram second. Any further optimization should target those phases based on this new evidence rather than the original transformation profile.

## Connection to LLMs

LLM systems combine several execution layers, each requiring different profiling evidence:

- Python orchestration, request handling, tokenization, scheduling, and post-processing;
- compiled CPU libraries;
- CUDA kernel launches and synchronization;
- GPU kernels;
- host-to-device, device-to-host, and inter-device transfers;
- distributed collectives;
- memory allocation and cache management.

A Python function profile can identify expensive orchestration paths, excessive calls, tokenization work, serialization, or synchronous control flow. It cannot determine whether a CUDA kernel has poor occupancy, whether tensor cores are used effectively, or whether execution is limited by device memory bandwidth.

Similarly, GPU kernel timing alone can miss Python launch overhead, request queueing, synchronization, data conversion, and communication.

The profiling workflow from this notebook generalizes to LLM workloads:

1. define the symptom and the measurement boundary;
2. establish an unprofiled end-to-end baseline;
3. choose a profiler appropriate to the suspected layer;
4. quantify profiler perturbation;
5. identify a bottleneck without over-interpreting the evidence;
6. change one relevant implementation detail;
7. validate output correctness;
8. measure end-to-end improvement;
9. profile the changed system again.

Bottleneck migration is common in inference systems. Accelerating attention kernels may expose sampling or tokenization overhead. Reducing Python launch overhead may expose memory bandwidth. Increasing single-GPU throughput may make networking or collective communication dominant.

## Further Exploration

Possible extensions include:

- vectorize the checksum and histogram, then re-profile the fully array-based pipeline;
- compare deterministic profiling with a statistical sampling profiler;
- run the Python workload under `perf stat` to collect cycles, instructions, branches, and cache-related counters;
- inspect the NumPy transformation with system-level sampling to determine where compiled execution occurs;
- vary workload size to separate fixed profiler cost from per-element cost;
- construct a call-heavy workload to show how deterministic-profiler overhead depends on call frequency;
- profile a CUDA workload with separate measurements for Python enqueue time, GPU execution time, and synchronized wall time;
- compare profiles before and after CUDA Graph capture;
- examine CPU–GPU synchronization points in an inference-style pipeline.